# Real-Time Eye State Detection (Open / Closed) for Driver Drowsiness

**MediaPipe Face Landmarker + MobileNetV3-Small** — the whole project in one notebook.

| Part | What happens |
|---|---|
| 0 | Setup: GPU, libraries, Google Drive, data |
| 1 | Our dataset: 5,452 eye images (MRL + colour images) |
| 2 | Person-level train / validation / test split |
| 3 | Leakage check (near-duplicate images) |
| 4 | Model, augmentation and training code |
| 5 | CEW external dataset: MediaPipe eye crops + EAR |
| 6 | Train the three models: **v2** (our data), **CEW-only**, **v3** (both, final) |
| 7 | Results on our test set and on the CEW test set, vs. EAR |
| 8 | Error analysis |
| 9 | Speed and model size |
| 10 | Test on a webcam photo |

**Runtime:** `Runtime → Change runtime type → T4 GPU`.
Set `RETRAIN = False` to load the saved models from Drive (fast); `True` trains everything again (~30–40 min).

## 0. Setup

In [ ]:
# ---------------- settings (change paths here if your Drive folder is different) ----------------
PROJECT    = "/content/drive/MyDrive/eye_project"
DATA_ZIP   = f"{PROJECT}/my_dataset/Mydataset.zip"                 # our dataset
CEW_ZIP    = f"{PROJECT}/my_dataset/dataset_B_Facial_Images.zip"   # CEW, 100 x 100 face images
SPLIT_FILE = f"{PROJECT}/split.csv"                                # person-level split (created in part 2 if missing)
MODELS     = f"{PROJECT}/models"
FACE_MODEL = f"{MODELS}/face_landmarker.task"
RESULTS    = f"{PROJECT}/results_final"                            # figures and numbers from this notebook

RETRAIN = False     # False: load saved models if they exist | True: train all models again
EPOCHS = 15
SEED = 0

In [ ]:
!pip install -q mediapipe

from google.colab import drive
drive.mount('/content/drive')

!unzip -q -o "{DATA_ZIP}" -d /content/my_dataset
!unzip -q -o "{CEW_ZIP}" -d /content/cew100

import os, urllib.request
os.makedirs(MODELS, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
if not os.path.exists(FACE_MODEL):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        FACE_MODEL)
print("Ready!")

In [ ]:
import csv, json, random, re, time
from collections import Counter, defaultdict
from math import atan2, degrees, hypot
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report, confusion_matrix,
                             roc_auc_score)
from sklearn.model_selection import train_test_split

import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

device = "cuda" if torch.cuda.is_available() else "cpu"
CLASSES = ["closed", "open"]          # label 0 = closed, label 1 = open
print("Device:", device)

## 1. Our dataset

5,452 pre-cropped eye images in two folders (`close`, `open`), from two sources:
**MRL Eye Dataset** (infrared, file names start with the person ID, e.g. `s0016_...`) and colour eye images from a second source.

In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
LABEL_OF_FOLDER = {"close": 0, "closed": 0, "open": 1}

samples = []      # (path, label, person)
for p in sorted(Path("/content/my_dataset").rglob("*")):
    folder = p.parent.name.lower()
    if p.suffix.lower() not in EXTS or folder not in LABEL_OF_FOLDER:
        continue
    m = re.match(r"(s\d{4})_", p.name)                 # MRL file names carry the person ID
    samples.append((str(p), LABEL_OF_FOLDER[folder], m.group(1) if m else "other"))

mrl = [s for s in samples if s[2] != "other"]
other = [s for s in samples if s[2] == "other"]
print(f"Total images: {len(samples)}  (closed {sum(s[1] == 0 for s in samples)}, open {sum(s[1] == 1 for s in samples)})")
print(f"MRL: {len(mrl)} images from {len({s[2] for s in mrl})} persons  |  second source: {len(other)} images")

fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, (path, label, person) in zip(axes.flat, random.Random(SEED).sample(samples, 12)):
    ax.imshow(Image.open(path).convert("RGB"))
    ax.set_title(f"{CLASSES[label]} | {person}", fontsize=9)
    ax.axis("off")
plt.suptitle("Samples from our dataset")
plt.tight_layout()
plt.savefig(f"{RESULTS}/dataset_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Person-level split

Frames of one person look almost the same, so a random **image** split would leak people into the test set.
We split the MRL images by **person** and choose, out of 20,000 random person assignments, the one whose
validation and test sets are closest to 15% of the data and to 50% closed eyes.
Second-source images have no person ID and are split at random.

If `split.csv` already exists on Drive it is **reused**, so every result in this notebook uses the same split.

In [ ]:
def make_split(seed=42):
    rng = random.Random(seed)
    other_split = {}
    for path, label, person in other:
        r = rng.random()
        other_split[path] = "test" if r < 0.15 else "val" if r < 0.30 else "train"

    counts = defaultdict(lambda: [0, 0])                 # person -> [closed, open]
    for path, label, person in mrl:
        counts[person][label] += 1
    persons = sorted(counts)
    base = {s: [0, 0] for s in ["train", "val", "test"]}
    for path, label, person in other:
        base[other_split[path]][label] += 1

    def cost(assign):
        totals = {s: list(base[s]) for s in base}
        for person, split in assign.items():
            totals[split][0] += counts[person][0]
            totals[split][1] += counts[person][1]
        c = 0.0
        for split in ["val", "test"]:
            closed, opened = totals[split]
            n = closed + opened
            c += abs(n / len(samples) - 0.15) + abs(closed / n - 0.5)
        return c

    best_cost, best = float("inf"), None
    for _ in range(20000):
        assign = {p: rng.choices(["train", "val", "test"], weights=[0.7, 0.15, 0.15])[0] for p in persons}
        if list(assign.values()).count("val") < 2 or list(assign.values()).count("test") < 2:
            continue
        c = cost(assign)
        if c < best_cost:
            best_cost, best = c, assign

    rows = [(path, label, person, other_split[path] if person == "other" else best[person])
            for path, label, person in samples]
    with open(SPLIT_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["path", "label", "person", "split"])
        writer.writerows(rows)

if Path(SPLIT_FILE).exists():
    print("Using the existing split:", SPLIT_FILE)
else:
    make_split()
    print("Created a new split:", SPLIT_FILE)

split_df = pd.read_csv(SPLIT_FILE)
table = split_df.groupby("split").agg(images=("label", "size"), closed=("label", lambda s: (s == 0).sum()),
                                      open=("label", lambda s: (s == 1).sum())).loc[["train", "val", "test"]]
table["share"] = (table["images"] / table["images"].sum()).map("{:.0%}".format)
display(table)
for s in ["val", "test"]:
    print(f"{s} persons:", sorted(p for p in split_df[split_df.split == s].person.unique() if p != "other"))

## 3. Leakage check

Each image gets a 16 × 16 black-and-white fingerprint (perceptual hash).
A validation or test image with the same fingerprint as a training image would be a near-duplicate.

In [ ]:
def fingerprint(path):
    a = np.asarray(Image.open(path).convert("L").resize((16, 16)), dtype=np.float32)
    return (a > a.mean()).tobytes()

prints = {path: fingerprint(path) for path in split_df.path}
train_prints = {prints[p] for p in split_df[split_df.split == "train"].path}
for s in ["val", "test"]:
    paths = split_df[split_df.split == s].path
    leaked = sum(prints[p] in train_prints for p in paths)
    print(f"{s}: {leaked} of {len(paths)} images have a near-identical twin in train")

## 4. Model, augmentation and training code

- **Model:** MobileNetV3-Small pretrained on ImageNet; last layer `1000 → 2` classes; BatchNorm momentum 0.1.
- **Input:** grayscale (3 channels), 112 × 112.
- **Augmentation (train only):** random crop, flip, rotation ±10°, brightness/contrast, Gaussian blur,
  and low-resolution simulation (shrink to 20–56 px and enlarge back, like small faces in CEW).
- **Training:** AdamW (lr 3e-4, weight decay 1e-4), cosine schedule, 15 epochs, keep the best epoch on validation.

In [ ]:
IMG_SIZE = 112
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class RandomLowRes:
    """Shrink to 20-56 px and enlarge back -> blurry, low-resolution look."""
    def __init__(self, p=0.5, min_size=20, max_size=56):
        self.p, self.min_size, self.max_size = p, min_size, max_size
    def __call__(self, img):
        if random.random() < self.p:
            s = random.randint(self.min_size, self.max_size)
            w, h = img.size
            img = img.resize((s, s), Image.BILINEAR).resize((w, h), Image.BILINEAR)
        return img

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    RandomLowRes(p=0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.5, contrast=0.5),
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class OurDataset(Dataset):
    """Our eye images, one split of split.csv."""
    def __init__(self, split, transform):
        rows = split_df[split_df.split == split]
        self.items = list(zip(rows.path, rows.label.astype(int)))
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        return self.transform(Image.open(path).convert("RGB")), label

def new_model(pretrained=True):
    m = models.mobilenet_v3_small(weights="IMAGENET1K_V1" if pretrained else None)
    m.classifier[3] = nn.Linear(1024, 2)
    for mod in m.modules():
        if isinstance(mod, nn.BatchNorm2d):
            mod.momentum = 0.1
    return m.to(device)

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            seen += y.size(0)
    return total_loss / seen, correct / seen

def train_model(train_set, val_set, save_path):
    torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
    model = new_model(pretrained=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=64, num_workers=2)
    history, best = [], -1.0
    for epoch in range(1, EPOCHS + 1):
        tl, ta = run_epoch(model, train_loader, optimizer)
        vl, va = run_epoch(model, val_loader)
        scheduler.step()
        history.append((tl, ta, vl, va))
        note = ""
        if va > best:
            best = va
            torch.save(model.state_dict(), save_path)
            note = "  <- best, saved"
        print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}{note}")
    model.load_state_dict(torch.load(save_path, map_location=device))
    return model.eval(), history

def get_model(name, train_set, val_set):
    """Load models/<name>.pt if it exists (and RETRAIN is False), else train and save it."""
    path = f"{MODELS}/{name}.pt"
    if Path(path).exists() and not RETRAIN:
        model = new_model(pretrained=False)
        model.load_state_dict(torch.load(path, map_location=device))
        print(f"Loaded {path}")
        return model.eval(), None
    print(f"Training {name} ...")
    return train_model(train_set, val_set, path)

def plot_history(history, title, file_name):
    if history is None:
        return
    e = range(1, len(history) + 1)
    h = np.array(history)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(e, h[:, 0], label="train"); a1.plot(e, h[:, 2], label="validation"); a1.set_title("Loss")
    a2.plot(e, h[:, 1], label="train"); a2.plot(e, h[:, 3], label="validation"); a2.set_title("Accuracy")
    for a in (a1, a2):
        a.set_xlabel("epoch"); a.legend()
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f"{RESULTS}/{file_name}", dpi=150, bbox_inches="tight")
    plt.show()

## 5. CEW external dataset: MediaPipe eye crops and EAR

**CEW (Closed Eyes in the Wild):** 2,423 real-world face images (1,192 closed, 1,231 open), 100 × 100 px.
Every face goes through the **same pipeline as the live demo**:
MediaPipe Face Landmarker → square crop around each eye (2.2 × eye width, rotated level) → model.

From the same landmarks we also compute the **Eye Aspect Ratio (EAR)**, the classic method with no training:
`EAR = (|p2 − p6| + |p3 − p5|) / (2 |p1 − p4|)`.

CEW is split by face: 70% for training (15% of it for validation) and **30% test, never used for training**.
CEW is for research use only and must not be redistributed.

In [ ]:
CROP_SIZE, CROP_SCALE = 128, 2.2
EYE_CORNERS = [(33, 133), (362, 263)]                  # (outer, inner) corner of each eye
RIGHT_EAR = [33, 160, 158, 133, 153, 144]               # corner, top, top, corner, bottom, bottom
LEFT_EAR = [362, 385, 387, 263, 373, 380]

landmarker = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_buffer=open(FACE_MODEL, "rb").read()),
    running_mode=vision.RunningMode.IMAGE, num_faces=1))

def eye_crops(frame_bgr, lm):
    """Square crop around each eye, rotated so the eye is level."""
    h, w = frame_bgr.shape[:2]
    crops = []
    for i1, i2 in EYE_CORNERS:
        p1 = np.array([lm[i1].x * w, lm[i1].y * h]); p2 = np.array([lm[i2].x * w, lm[i2].y * h])
        if p1[0] > p2[0]:
            p1, p2 = p2, p1
        c = (p1 + p2) / 2
        side = hypot(*(p2 - p1)) * CROP_SCALE
        M = cv2.getRotationMatrix2D((float(c[0]), float(c[1])), degrees(atan2(p2[1] - p1[1], p2[0] - p1[0])), CROP_SIZE / side)
        M[0, 2] += CROP_SIZE / 2 - c[0]; M[1, 2] += CROP_SIZE / 2 - c[1]
        crops.append(cv2.warpAffine(frame_bgr, M, (CROP_SIZE, CROP_SIZE), borderMode=cv2.BORDER_REPLICATE))
    return crops

def eye_aspect_ratio(lm, idx, w, h):
    p = [np.array([lm[i].x * w, lm[i].y * h]) for i in idx]
    return (np.linalg.norm(p[1] - p[5]) + np.linalg.norm(p[2] - p[4])) / (2 * np.linalg.norm(p[0] - p[3]))

def prepare_face(frame_bgr):
    """CEW faces are tightly cropped and small: add a grey border, then upscale for MediaPipe."""
    pad = int(0.4 * max(frame_bgr.shape[:2]))
    frame = cv2.copyMakeBorder(frame_bgr, pad, pad, pad, pad, cv2.BORDER_CONSTANT, value=(128, 128, 128))
    if max(frame.shape[:2]) < 400:
        s = 400 / max(frame.shape[:2])
        frame = cv2.resize(frame, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
    return frame

CEW_DIRS = {0: next(Path("/content/cew100").rglob("ClosedFace")), 1: next(Path("/content/cew100").rglob("OpenFace"))}

items = []          # (label, [right_crop, left_crop], EAR)
no_face = 0
for label, folder in CEW_DIRS.items():
    for path in sorted(folder.glob("*")):
        frame = cv2.imread(str(path))
        if frame is None:
            continue
        frame = prepare_face(frame)
        res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
        if not res.face_landmarks:
            no_face += 1
            continue
        lm = res.face_landmarks[0]
        h, w = frame.shape[:2]
        ear = (eye_aspect_ratio(lm, RIGHT_EAR, w, h) + eye_aspect_ratio(lm, LEFT_EAR, w, h)) / 2
        items.append((label, eye_crops(frame, lm), ear))

cew_labels = np.array([it[0] for it in items])
cew_ears = np.array([it[2] for it in items])
print(f"CEW faces found: {len(items)}  (no face found: {no_face})")

idx = np.arange(len(items))
cew_train_idx, cew_test_idx = train_test_split(idx, test_size=0.30, stratify=cew_labels, random_state=0)
cew_fit_idx, cew_val_idx = train_test_split(cew_train_idx, test_size=0.15, stratify=cew_labels[cew_train_idx], random_state=0)
print(f"CEW split: train {len(cew_fit_idx)} faces, validation {len(cew_val_idx)}, TEST {len(cew_test_idx)}")

class CewCrops(Dataset):
    """Eye crops (2 per face) of the given CEW faces."""
    def __init__(self, ids, transform):
        self.samples = [(crop, items[i][0]) for i in ids for crop in items[i][1]]
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, k):
        crop, label = self.samples[k]
        return self.transform(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))), label

## 6. Train the models

| Model | Trained on | Why |
|---|---|---|
| **v2** | our data only | the model before CEW |
| **CEW-only** | 70% of CEW only | the same model trained like most papers (train and test on one dataset) |
| **v3 (final)** | our data + 70% of CEW | varied data, used in the live demo |

(The very first model, v1, was the same as v2 without the blur / low-resolution augmentation:
99.79% on our test set, 87.85% on all of CEW.)

In [ ]:
model_v2, hist_v2 = get_model("eye_state_mnv3s_v2",
                              OurDataset("train", train_transform), OurDataset("val", eval_transform))
plot_history(hist_v2, "Model v2 (our data)", "training_curves_v2.png")

In [ ]:
model_cew, hist_cew = get_model("eye_state_mnv3s_cew",
                                CewCrops(cew_fit_idx, train_transform), CewCrops(cew_val_idx, eval_transform))
plot_history(hist_cew, "CEW-only model", "training_curves_cew.png")

In [ ]:
model_v3, hist_v3 = get_model("eye_state_mnv3s_v3",
                              ConcatDataset([OurDataset("train", train_transform), CewCrops(cew_fit_idx, train_transform)]),
                              ConcatDataset([OurDataset("val", eval_transform), CewCrops(cew_val_idx, eval_transform)]))
plot_history(hist_v3, "Model v3 (our data + CEW)", "training_curves_v3.png")

## 7. Results

- **Our test set:** 971 eye images of persons never seen in training.
- **CEW test set:** 726 faces (30% of CEW), never used for training. A face counts as open if the average P(open) of both eyes is above 0.5.
- **EAR** needs a full face, so it can only be tested on CEW. Its threshold is tuned on the CEW training faces.

In [ ]:
our_test = OurDataset("test", eval_transform)
y_our = np.array([label for _, label in our_test.items])
y_cew = cew_labels[cew_test_idx]

def probs_our(model):
    model.eval()
    out = []
    with torch.no_grad():
        for x, _ in DataLoader(our_test, batch_size=128, num_workers=2):
            out.append(torch.softmax(model(x.to(device)), 1)[:, 1].cpu())
    return torch.cat(out).numpy()

def probs_cew(model, ids):
    model.eval()
    crops = [crop for i in ids for crop in items[i][1]]
    x = torch.stack([eval_transform(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in crops])
    out = []
    with torch.no_grad():
        for k in range(0, len(x), 128):
            out.append(torch.softmax(model(x[k:k + 128].to(device)), 1)[:, 1].cpu())
    return torch.cat(out).numpy().reshape(-1, 2).mean(1)        # average of both eyes

def scores(y, p_open, threshold=0.5):
    pred = (p_open > threshold).astype(int)
    return {"accuracy": (pred == y).mean(),
            "closed recall": ((pred == 0) & (y == 0)).sum() / (y == 0).sum(),
            "open recall": ((pred == 1) & (y == 1)).sum() / (y == 1).sum(),
            "AUC": roc_auc_score(y, p_open)}

P_our = {name: probs_our(m) for name, m in [("v2", model_v2), ("CEW-only", model_cew), ("v3", model_v3)]}
P_cew = {name: probs_cew(m, cew_test_idx) for name, m in [("v2", model_v2), ("CEW-only", model_cew), ("v3", model_v3)]}

thresholds = np.arange(0.05, 0.45, 0.005)
ear_t = max(thresholds, key=lambda t: ((cew_ears[cew_train_idx] > t).astype(int) == cew_labels[cew_train_idx]).mean())
ear_test = cew_ears[cew_test_idx]

rows = []
for name, trained_on in [("v2", "our data"), ("CEW-only", "CEW 70%"), ("v3", "our data + CEW")]:
    a, b = scores(y_our, P_our[name]), scores(y_cew, P_cew[name])
    rows.append({"method": f"CNN {name}", "trained on": trained_on,
                 "our test acc": a["accuracy"], "CEW test acc": b["accuracy"],
                 "CEW closed recall": b["closed recall"], "CEW open recall": b["open recall"], "CEW AUC": b["AUC"]})
for t, label in [(0.20, "EAR, threshold 0.20"), (ear_t, f"EAR, threshold {ear_t:.3f} (tuned)")]:
    b = scores(y_cew, ear_test, t)
    rows.append({"method": label, "trained on": "no training", "our test acc": np.nan, "CEW test acc": b["accuracy"],
                 "CEW closed recall": b["closed recall"], "CEW open recall": b["open recall"], "CEW AUC": b["AUC"]})

results = pd.DataFrame(rows).set_index("method")
display(results.style.format("{:.2%}", subset=["our test acc", "CEW test acc", "CEW closed recall", "CEW open recall"],
                             na_rep="needs full face").format("{:.4f}", subset=["CEW AUC"]))
results.to_csv(f"{RESULTS}/results_table.csv")

In [ ]:
# Final model v3: full report and confusion matrices
pred_our = (P_our["v3"] > 0.5).astype(int)
pred_cew = (P_cew["v3"] > 0.5).astype(int)
print("=== Model v3, OUR TEST SET ===")
print(classification_report(y_our, pred_our, target_names=CLASSES, digits=4, zero_division=0))
print("=== Model v3, CEW TEST SET ===")
print(classification_report(y_cew, pred_cew, target_names=CLASSES, digits=4, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix(y_our, pred_our), display_labels=CLASSES).plot(ax=axes[0], cmap="Blues", values_format="d", colorbar=False)
axes[0].set_title(f"v3, our test set ({len(y_our)} eye images)")
ConfusionMatrixDisplay(confusion_matrix(y_cew, pred_cew), display_labels=CLASSES).plot(ax=axes[1], cmap="Greens", values_format="d", colorbar=False)
axes[1].set_title(f"v3, CEW test set ({len(y_cew)} faces)")
plt.tight_layout()
plt.savefig(f"{RESULTS}/confusion_matrices_v3.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# How well does EAR separate open from closed faces on CEW?
plt.figure(figsize=(8, 4))
plt.hist(cew_ears[cew_labels == 0], bins=50, alpha=0.6, label="closed")
plt.hist(cew_ears[cew_labels == 1], bins=50, alpha=0.6, label="open")
plt.axvline(0.20, color="k", linestyle="--", label="threshold 0.20")
plt.axvline(ear_t, color="r", linestyle=":", label=f"tuned threshold {ear_t:.3f}")
plt.xlabel("EAR"); plt.ylabel("number of faces"); plt.title("EAR distribution on CEW"); plt.legend()
plt.savefig(f"{RESULTS}/ear_histogram_cew.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the results:** each CNN is strong on data like its own training data and weaker on the other dataset
(**domain shift**, in both directions). Trained on both, **v3** stays high on our test set and beats EAR on CEW.

## 8. Error analysis (model v3)

In [ ]:
def show_errors(images, titles, header, n=8):
    if not images:
        print(header, "- no errors")
        return
    n = min(n, len(images))
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.8), squeeze=False)
    for ax, img, t in zip(axes[0], images[:n], titles[:n]):
        ax.imshow(img); ax.set_title(t, fontsize=8); ax.axis("off")
    plt.suptitle(header)
    plt.show()

wrong = np.where(pred_our != y_our)[0]
show_errors([Image.open(our_test.items[i][0]).convert("RGB") for i in wrong],
            [f"true {CLASSES[y_our[i]]}\nP(open)={P_our['v3'][i]:.2f}" for i in wrong],
            f"Our test set: {len(wrong)} errors")

wrong = np.where(pred_cew != y_cew)[0]
show_errors([cv2.cvtColor(np.hstack(items[cew_test_idx[i]][1]), cv2.COLOR_BGR2RGB) for i in wrong],
            [f"true {CLASSES[y_cew[i]]}\nP(open)={P_cew['v3'][i]:.2f}" for i in wrong],
            f"CEW test set: {len(wrong)} errors (both eye crops shown)")

## 9. Speed and model size

Time for one video frame (two eye crops). Speed depends on the model's structure, not on its weights,
so the comparison models are measured untrained, with their original 1000-class layer (ours too, for a fair comparison).
On a laptop CPU (AMD Ryzen, no GPU) the **full pipeline** (MediaPipe + crops + model) measured **46.8 ms ≈ 21 FPS** with `benchmark.py`.

In [ ]:
def ms_per_frame(model, dev, runs=100):
    model = model.to(dev).eval()
    x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=dev)
    with torch.no_grad():
        for _ in range(10):
            model(x)
        if dev == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(runs):
            model(x)
        if dev == "cuda":
            torch.cuda.synchronize()
    return (time.perf_counter() - start) / runs * 1000

size_mb = Path(f"{MODELS}/eye_state_mnv3s_v3.pt").stat().st_size / 1e6
params = sum(p.numel() for p in model_v3.parameters())
cpu = ms_per_frame(model_v3, "cpu")
print(f"Model v3: {params / 1e6:.2f} M parameters, file {size_mb:.1f} MB")
print(f"CPU: {cpu:.1f} ms per frame -> {1000 / cpu:.0f} FPS")
if torch.cuda.is_available():
    gpu = ms_per_frame(model_v3, "cuda")
    print(f"GPU: {gpu:.1f} ms per frame -> {1000 / gpu:.0f} FPS")
model_v3.to(device)

rows = []
for name, m in [("MobileNetV3-Small (ours)", models.mobilenet_v3_small()), ("MobileNetV3-Large", models.mobilenet_v3_large()),
                ("EfficientNet-B0", models.efficientnet_b0()), ("ResNet-18", models.resnet18()), ("ResNet-50", models.resnet50())]:
    p = sum(q.numel() for q in m.parameters())
    t = ms_per_frame(m, "cpu", runs=50)
    rows.append({"model": name, "params (M)": round(p / 1e6, 2), "size (MB)": round(p * 4 / 1e6, 1),
                 "CPU ms / frame": round(t, 1), "CPU FPS": round(1000 / t)})
speed = pd.DataFrame(rows).set_index("model")
display(speed)
speed.to_csv(f"{RESULTS}/speed_comparison.csv")

## 10. Test on a webcam photo

Press **Capture**; the photo is taken 3 seconds later (enough time to close your eyes).
The decision uses the average P(open) of both eyes; EAR is shown for comparison.

In [ ]:
from IPython.display import Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename="photo.jpg", delay_ms=3000):
    display(Javascript('''
    async function takePhoto(delayMs) {
      const div = document.createElement('div');
      const button = document.createElement('button');
      button.textContent = 'Capture (photo after 3 seconds)';
      div.appendChild(button);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((resolve) => button.onclick = resolve);
      button.textContent = 'Taking photo in 3 seconds...';
      await new Promise((resolve) => setTimeout(resolve, delayMs));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', 0.9);
    }
    '''))
    data = eval_js(f"takePhoto({delay_ms})")
    with open(filename, "wb") as f:
        f.write(b64decode(data.split(",")[1]))
    return filename

def analyze(image_path, model=model_v3):
    frame = cv2.imread(image_path)
    res = landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    if not res.face_landmarks:
        print("No face found!")
        return
    lm = res.face_landmarks[0]
    h, w = frame.shape[:2]
    crops = eye_crops(frame, lm)
    x = torch.stack([eval_transform(Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))) for c in crops]).to(device)
    with torch.no_grad():
        p_open = torch.softmax(model.eval()(x), 1)[:, 1].cpu().numpy()
    p_avg = float(p_open.mean())
    ear = (eye_aspect_ratio(lm, RIGHT_EAR, w, h) + eye_aspect_ratio(lm, LEFT_EAR, w, h)) / 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), gridspec_kw={"width_ratios": [3, 1, 1]})
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"FINAL: {'OPEN' if p_avg > 0.5 else 'CLOSED'}  ({p_avg:.3f})   |   EAR = {ear:.3f}", fontsize=13)
    for ax, crop, name, p in zip(axes[1:], crops, ["right eye", "left eye"], p_open):
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{name}\n{'OPEN' if p > 0.5 else 'CLOSED'}  ({p:.3f})")
    for ax in axes:
        ax.axis("off")
    plt.show()

analyze(take_photo())

## Summary

- Person-level split, 0 near-duplicates between train and val/test.
- Final model **v3** (MobileNetV3-Small, 1.52 M parameters): high accuracy on our test set **and** on the unseen CEW test faces, above the EAR baseline.
- Real time on a laptop CPU (~21 FPS full pipeline). Live demo: `demo.py` (GitHub repository).